# Treino e deployment do classificador de ransomware

Este notebook executa o benchmark scikit-learn, consulta as métricas guardadas e invoca a mesma rotina Python usada pela CLI/API para prever linhas de exemplo. As linhas de demonstração servem apenas para validar o fluxo de inferência; não são uma avaliação externa.

**Limite:** o modelo recebe características já extraídas. Não abre, executa nem extrai características de ficheiros `.exe`.

## 1. Preparar o projeto e os dados

Executa o notebook com o kernel do ambiente `.venv` do projeto.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "ransom.csv").is_file():
    PROJECT_ROOT = next(
        (parent for parent in PROJECT_ROOT.parents if (parent / "ransom.csv").is_file()),
        PROJECT_ROOT,
    )
DATA_PATH = PROJECT_ROOT / "ransom.csv"
OUTPUT_DIR = PROJECT_ROOT / "results"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset não encontrado: {DATA_PATH}")
sys.path.insert(0, str(PROJECT_ROOT))

from src.supervised_learning import train_and_compare
from src.predict import predict_all_models, predict_file

print(f"Projeto: {PROJECT_ROOT}")
print(f"Dataset: {DATA_PATH}")
print(f"Dimensão: {pd.read_csv(DATA_PATH).shape}")

## 2. Treinar, comparar e selecionar

A seleção é feita com recall de Malware na validação, com precision e F1 como desempate. O teste é reservado à medição final. Todos os classificadores finais e o schema são guardados em `models/ransomware/`.

In [ ]:
MODEL_DIR = PROJECT_ROOT / "models" / "ransomware"
comparison = train_and_compare(DATA_PATH, OUTPUT_DIR, model_dir=MODEL_DIR)
display(comparison)

report = json.loads(
    (OUTPUT_DIR / "supervised_learning_report.json").read_text(encoding="utf-8")
)
print("Modelo selecionado:", report["selected_model"])
print("Métricas no teste reservado:")
display(pd.Series(report["final_test_metrics"], name="test"))

## 3. Invocar a inferência Python

A amostra abaixo é uma demonstração de interface, não uma validação independente: foi retirada do dataset conhecido. O modelo só recebe as features do schema guardado.

In [ ]:
from tempfile import TemporaryDirectory

model_path = MODEL_DIR / f"{report['selected_model']}.joblib"
with TemporaryDirectory(prefix="ransomware-demo-") as temporary_directory:
    sample_path = Path(temporary_directory) / "sample.csv"
    pd.read_csv(DATA_PATH, nrows=3).to_csv(sample_path, index=False)
    predictions = predict_file(model_path, sample_path)
    all_model_predictions = predict_all_models(MODEL_DIR, sample_path)

display(predictions)
display(all_model_predictions)

## 4. Deployment local

Depois de executar o treino, inicia `python src/serve.py` num terminal. A API fica em `http://127.0.0.1:8000`; consulta `docs/deployment.md` para `/health`, `/predict`, formato JSON e limites do protótipo.

## 5. Invocar os agentes generativos

Instala primeiro `python -m pip install -r requirements-ai.txt`. O primeiro caminho usa o orquestrador Python; o segundo usa CrewAI. Ambos reutilizam o melhor classificador e o mesmo modelo Hugging Face local, e guardam relatórios assistidos em `reports/`. Nenhum dado é enviado a serviços externos.

In [ ]:
import gc
from src.agent_system import (
    GenerativeReportAgent,
    create_agent_facts,
    write_agent_report,
)
from src.crewai_agents import run_crewai_report_async

with TemporaryDirectory(prefix="ransomware-agents-") as temporary_directory:
    agent_input = Path(temporary_directory) / "sample.csv"
    pd.read_csv(DATA_PATH, nrows=3).to_csv(agent_input, index=False)
    facts = create_agent_facts(
        agent_input,
        OUTPUT_DIR / "supervised_learning_report.json",
        MODEL_DIR,
    )

    local_agent = GenerativeReportAgent(cache_dir=PROJECT_ROOT / "models" / "huggingface")
    local_narrative = local_agent.run(facts)
    write_agent_report(
        facts, local_narrative, PROJECT_ROOT / "reports" / "ransomware-local.md"
    )
    del local_agent
    gc.collect()

    crewai_narrative = await run_crewai_report_async(
        facts, cache_dir=PROJECT_ROOT / "models" / "huggingface"
    )
    write_agent_report(
        facts, crewai_narrative, PROJECT_ROOT / "reports" / "ransomware-crewai.md"
    )

print("Relatórios assistidos: reports/ransomware-local.md e reports/ransomware-crewai.md")